In [1]:
import os
import pandas as pd
from PIL import Image

## Noisy dataset clustering

Scan noisy images, compute DINOv1 embeddings, find similarity pairs via FAISS.

Uses image **filename** as `image_id` (unique across all classes).

In [2]:
noisy_data_path = "/srv/defectDetectionDataset/surfaceClassification/noisy"
artifacts_path = "./artifacts/noisy"
os.makedirs(artifacts_path, exist_ok=True)

In [3]:
noisy_images = []
extensions = [".png", ".jpg", ".jpeg"]

for dirpath, dirnames, filenames in os.walk(noisy_data_path):
    for filename in filenames:
        if any(filename.lower().endswith(ext) for ext in extensions):
            file_path = os.path.join(dirpath, filename)
            label = os.path.basename(dirpath)
            rel_path = file_path.removeprefix(noisy_data_path)
            image_id = filename
            noisy_images.append({"image_id": image_id, "file_path": rel_path, "label": label})

df = pd.DataFrame(noisy_images)
df.to_csv(os.path.join(artifacts_path, "noisy_images.csv"), index=False)

print(f"Number of images: {len(df)}")
print(f"Label distribution:")
print(df["label"].value_counts())

Number of images: 5475
Label distribution:
label
ok                1955
deformation       1230
corrosion          590
water_stain        395
background         342
missing_part       324
crack              319
other              195
silicate_stain      89
black_stain         36
Name: count, dtype: int64


## Compute DINOv1 embeddings

In [5]:
import torch
import numpy as np
from torchvision import transforms
from tqdm import tqdm
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

device = "cuda" if torch.cuda.is_available() else "cpu"

model = torch.hub.load("facebookresearch/dino:main", "dino_vits16")
model.to(device).eval()

preprocess = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


@torch.no_grad()
def get_dino_embedding(img):
    img_tensor = preprocess(img).unsqueeze(0).to(device)
    feats = model(img_tensor)
    return torch.nn.functional.normalize(feats, dim=-1)


embeddings = []
for row in tqdm(df.itertuples(index=False), total=len(df)):
    file_path = noisy_data_path + row.file_path
    img = Image.open(file_path).convert("RGB")
    embedding = get_dino_embedding(img).cpu().numpy().flatten()
    embeddings.append(embedding)

embeddings_array = np.array(embeddings)
np.save(os.path.join(artifacts_path, "image_embeddings.npy"), embeddings_array)
df[["image_id"]].to_csv(os.path.join(artifacts_path, "image_embeddings_ids.csv"), index=False)

print(f"Generated embeddings for {len(embeddings)} images")
print(f"Embedding shape: {embeddings_array.shape}")

Downloading: "https://github.com/facebookresearch/dino/zipball/main" to /home/researcher/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth" to /home/researcher/.cache/torch/hub/checkpoints/dino_deitsmall16_pretrain.pth


100%|██████████| 82.7M/82.7M [00:27<00:00, 3.13MB/s]
100%|██████████| 5475/5475 [00:33<00:00, 165.26it/s]

Generated embeddings for 5475 images
Embedding shape: (5475, 384)


## FAISS nearest-neighbor search

In [6]:
import faiss

dimension = embeddings_array.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_array)
print(f"FAISS index built with {index.ntotal} vectors of dimension {dimension}")

k = 20
D, I = index.search(embeddings_array, k)
D, I = torch.from_numpy(D), torch.from_numpy(I)

FAISS index built with 5475 vectors of dimension 384


## Build similarity pairs

In [7]:
image_ids = df["image_id"].tolist()

# Convert L2 distances to cosine similarity (vectors are unit-normalized)
similarities = 1 - (D / 2)
similarities = torch.clamp(similarities, -1.0, 1.0)

similarity_pairs = []
seen_pairs = set()
for i in range(len(similarities)):
    for j in range(1, k):  # skip self (index 0)
        img_id_1 = image_ids[i]
        img_id_2 = image_ids[I[i][j].item()]
        sim_score = similarities[i][j].item()
        pair_key = tuple(sorted([img_id_1, img_id_2]))
        if pair_key in seen_pairs:
            continue
        seen_pairs.add(pair_key)
        similarity_pairs.append({
            "image_id_1": img_id_1,
            "image_id_2": img_id_2,
            "similarity": sim_score,
        })

similarity_df = pd.DataFrame(similarity_pairs)
similarity_df.to_csv(os.path.join(artifacts_path, "similarity_pairs.csv"), index=False)
print(f"Saved {len(similarity_df)} similarity pairs")

Saved 71204 similarity pairs
